# Diagnosing two numerical problems in fracridge

## Executive summary

This notebook provides executable evidence for two defects in the current
implementation. It calls the package as it exists: no monkeypatching and no
replacement of package internals. Local reference functions serve only as
controls that isolate each proposed correction; they are not proposed public
APIs.

The two findings are:

1. `FracRidgeRegressorCV` preprocesses the complete dataset before
   cross-validation and later predicts raw inputs with an intercept learned in
   centered coordinates.
2. `_do_svd` decomposes `X.T @ X` for tall designs, squaring the condition
   number and producing inaccurate coefficients on nearly collinear data.

Fraction `1.0` is used to isolate both defects from intentional ridge
shrinkage. Assertions make the numerical claims executable. The notebook is
committed without outputs so reviewers can reproduce the evidence locally.

## Problem 1: cross-validation loses the intercept

### Expected invariant: affine OLS behavior

The deterministic data below obey `y = 3 + 2X` exactly. With the only candidate
fraction set to `1.0`, fractional ridge should reduce to the unregularized
solution: slope `2`, intercept `3`, maximum prediction error near machine
precision, and R-squared `1`. A constant prediction offset therefore cannot be
attributed to intended coefficient shrinkage.

### Current code path: preprocessing in the wrong coordinate system

The failure begins in `FracRidgeRegressorCV.fit`. Its call to
`self._validate_input(X, y)` invokes `_preprocess_data`, centering the complete
dataset when `fit_intercept=True`. Those already-centered arrays are then given
to `GridSearchCV`, whose inner `FracRidgeRegressor` centers every fold again.

After the final refit, the outer estimator copies `coef_` and `intercept_` from
`best_estimator_`. That intercept is expressed relative to centered inputs, but
the inherited `predict` method is later called with raw `X`. The coordinate
offset removed before cross-validation is never restored. Centering before the
fold split also lets whole-dataset offsets influence every fold's input.

The next cell uses a fixed shuffled split and a single fraction to make this
causal chain visible without model-selection ambiguity.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import r2_score
from sklearn.model_selection import GridSearchCV, KFold

from fracridge.fracridge import (
    FracRidgeRegressor,
    FracRidgeRegressorCV,
    fracridge,
)


In [ ]:
X_intercept = np.linspace(0.0, 1.0, 101)[:, np.newaxis]
y_intercept = 3.0 + 2.0 * X_intercept[:, 0]
cv = KFold(n_splits=5, shuffle=True, random_state=20260804)

current_cv = FracRidgeRegressorCV(
    fit_intercept=True,
    cv=cv,
    jit=False,
)
current_cv.fit(X_intercept, y_intercept, frac_grid=np.array([1.0]))
current_prediction = current_cv.predict(X_intercept)
current_intercept = float(np.asarray(current_cv.intercept_).squeeze())
current_slope = float(np.asarray(current_cv.coef_).squeeze())
current_max_error = float(np.max(np.abs(current_prediction - y_intercept)))
current_r2 = r2_score(y_intercept, current_prediction)

print(f"learned intercept: {current_intercept:.12g}")
print(f"learned slope: {current_slope:.12g}")
print(f"maximum prediction error: {current_max_error:.12g}")
print(f"R-squared: {current_r2:.12g}")

assert abs(current_intercept - 3.0) > 2.5
assert current_max_error > 2.5


### Interpreting the current estimator

The learned slope remains `2`, showing that the estimator identifies the
relationship between `X` and `y`. The learned intercept is instead near zero,
and every prediction is displaced by approximately `3`. This is a
coordinate-system mismatch, not a failure to estimate the slope. The assertions
deliberately encode that current failure so this notebook stops demonstrating
the defect once the package is corrected.

### Executable reference: let each estimator center raw data once

### Why the raw-data reference isolates the fix

The reference changes one boundary: raw `X` and `y` enter `GridSearchCV`.
Consequently, each training fold computes its own offsets exactly once, and the
final refit computes offsets from the complete raw training data exactly once.
The fitted estimator then predicts in the same coordinate system in which its
intercept was defined.

This is an executable control, not the final package patch. If its intercept,
prediction error, and R-squared recover while the current CV estimator fails on
the same data and folds, preprocessing placement is the isolated cause.

In [ ]:
def fit_cv_reference(X, y, *, frac_grid, cv):
    estimator = FracRidgeRegressor(fit_intercept=True, jit=False)
    search = GridSearchCV(estimator, {"fracs": np.asarray(frac_grid)}, cv=cv)
    return search.fit(X, y)


reference_cv = fit_cv_reference(
    X_intercept,
    y_intercept,
    frac_grid=np.array([1.0]),
    cv=cv,
)
reference_estimator = reference_cv.best_estimator_
reference_prediction = reference_estimator.predict(X_intercept)
reference_intercept = float(np.asarray(reference_estimator.intercept_).squeeze())
reference_slope = float(np.asarray(reference_estimator.coef_).squeeze())
reference_max_error = float(
    np.max(np.abs(reference_prediction - y_intercept))
)
reference_r2 = r2_score(y_intercept, reference_prediction)

print(f"reference intercept: {reference_intercept:.12g}")
print(f"reference slope: {reference_slope:.12g}")
print(f"reference maximum prediction error: {reference_max_error:.12g}")
print(f"reference R-squared: {reference_r2:.12g}")

assert np.isclose(reference_intercept, 3.0, atol=1e-10)
assert reference_max_error < 1e-10


The reference restores intercept `3`, machine-precision prediction error, and
R-squared `1`. The overlaid plot makes the consequence concrete: the current
predictions have the correct slope but the wrong vertical origin, while the
raw-data reference coincides with the truth.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(X_intercept[:, 0], y_intercept, label="truth", linewidth=3)
ax.plot(
    X_intercept[:, 0],
    current_prediction,
    "--",
    label="current FracRidgeRegressorCV",
)
ax.plot(
    X_intercept[:, 0],
    reference_prediction,
    ":",
    label="raw-data GridSearchCV reference",
    linewidth=3,
)
ax.set(xlabel="X", ylabel="prediction", title="Intercept handling")
ax.legend()
fig.tight_layout()
plt.show()


### Problem 1 acceptance criteria

The production fix should validate input shape without centering the complete
dataset, pass raw validated inputs to `GridSearchCV`, and copy the fitted state
from `best_estimator_` consistently. A regression test must verify nonzero-offset
affine data and must be committed in a failing state before implementation.

The patch must also preserve feature-count and fitted-state metadata, behavior
when `fit_intercept=False`, and multi-target behavior. The reference above shows
the required preprocessing boundary; it is not intended to dictate a new public
API.

## Problem 2: normal equations amplify numerical error

### Expected invariant: stable OLS at fraction 1.0

At fraction `1.0`, fractional ridge should reproduce a stable unregularized
least-squares solution. The comparisons below use `np.linalg.lstsq`, which
solves from `X` directly, as the numerical reference.

### Current code path: normal equations square conditioning

For a tall design, `_do_svd` currently decomposes `X.T @ X` rather than `X`.
For the nonzero singular spectrum,

```text
cond(X.T @ X) = cond(X)^2
```

Squaring the condition number consumes roughly twice as many decimal digits of
precision. Errors in the recovered singular system are therefore present before
fracridge computes its shrinkage factors or rotates coefficients back to the
original basis.

The experiment below creates that failure in a controlled way by making two
columns progressively more collinear while holding every random draw fixed.

### Executable reference: decompose X directly

### Why the direct-SVD reference solves the root cause

The local reference computes an economy SVD of `X` itself, removes singular
directions below the tolerance before division, rotates `y` into the left
singular basis, and constructs the OLS coordinates there. For fractions below
one, a monotonic bisection finds the nonnegative alpha whose shrunk coefficient
norm matches the requested fraction. The coefficients are then rotated back.

This diagnostic reference intentionally accepts only a one-dimensional response.
The production implementation must preserve vector, multi-target, tall, and
wide shapes.

In [ ]:
def coefficient_norm_at_alpha(singular_values, ols_rotated, alpha):
    singular_values_sq = singular_values**2
    shrinkage = singular_values_sq / (singular_values_sq + alpha)
    return float(np.linalg.norm(shrinkage * ols_rotated))


def solve_fraction_alpha(singular_values, ols_rotated, fraction):
    if not 0.0 < fraction <= 1.0:
        raise ValueError("fraction must be in the interval (0, 1]")
    if fraction == 1.0:
        return 0.0

    target_norm = fraction * np.linalg.norm(ols_rotated)
    lower = 0.0
    upper = float(singular_values[0] ** 2)
    while coefficient_norm_at_alpha(singular_values, ols_rotated, upper) > target_norm:
        upper *= 2.0

    for _ in range(100):
        midpoint = (lower + upper) / 2.0
        midpoint_norm = coefficient_norm_at_alpha(
            singular_values, ols_rotated, midpoint
        )
        if midpoint_norm > target_norm:
            lower = midpoint
        else:
            upper = midpoint
    return (lower + upper) / 2.0


def direct_svd_components(X, y, tol):
    if np.ndim(y) != 1:
        raise ValueError("y must be one-dimensional")

    left, singular_values, right_transpose = np.linalg.svd(
        X, full_matrices=False
    )
    active = singular_values > tol
    if not np.any(active):
        raise ValueError("X has no singular values above tol")
    projected_y = left[:, active].T @ y
    ols_rotated = projected_y / singular_values[active]
    return singular_values[active], right_transpose[active], ols_rotated


def fracridge_direct_svd(X, y, fraction, *, tol=1e-10):
    singular_values, right_transpose, ols_rotated = direct_svd_components(
        X, y, tol
    )
    alpha = solve_fraction_alpha(singular_values, ols_rotated, fraction)
    singular_values_sq = singular_values**2
    shrinkage = singular_values_sq / (singular_values_sq + alpha)
    coefficients = right_transpose.T @ (shrinkage * ols_rotated)
    return coefficients, alpha


### Controlled experiment and reported metrics

Only `epsilon`, the perturbation separating the first two columns, changes
between rows. Smaller epsilon produces a larger `cond(X)`. For every design the
table reports:

- **current error:** coefficient disagreement between current fracridge at
  fraction `1.0` and `np.linalg.lstsq`;
- **direct-SVD error:** the same disagreement for the local reference;
- **current residual** and **direct-SVD residual:** the norms of `X @ coef - y`.

The coefficient errors test whether the solver recovered the stable parameter
vector. Residuals test only whether the fitted predictions reproduce this
particular response.

In [ ]:
def make_tall_design(base, perturbation, extras, epsilon):
    near_duplicate = base + epsilon * perturbation
    return np.column_stack((base, near_duplicate, extras))


rng = np.random.default_rng(20260804)
n_samples = 240
base_column = rng.normal(size=n_samples)
perturbation = rng.normal(size=n_samples)
extra_columns = rng.normal(size=(n_samples, 3))
true_coefficients = np.array([1.0, -1.0, 0.5, -0.25, 0.125])
epsilons = np.logspace(-4, -8, 5)
records = []

for epsilon in epsilons:
    design = make_tall_design(
        base_column, perturbation, extra_columns, epsilon
    )
    response = design @ true_coefficients
    least_squares = np.linalg.lstsq(design, response, rcond=None)[0]
    current_coefficients, _ = fracridge(
        design, response, fracs=1.0, jit=False
    )
    reference_coefficients, _ = fracridge_direct_svd(
        design, response, fraction=1.0
    )
    denominator = np.linalg.norm(least_squares)
    records.append(
        {
            "epsilon": epsilon,
            "condition": np.linalg.cond(design),
            "current_error": np.linalg.norm(
                current_coefficients - least_squares
            )
            / denominator,
            "reference_error": np.linalg.norm(
                reference_coefficients - least_squares
            )
            / denominator,
            "current_residual": np.linalg.norm(
                design @ current_coefficients - response
            ),
            "reference_residual": np.linalg.norm(
                design @ reference_coefficients - response
            ),
        }
    )

print(
    f"{'epsilon':>10} {'cond(X)':>14} {'current error':>16} "
    f"{'direct-SVD error':>18} {'current residual':>18} "
    f"{'direct-SVD residual':>20}"
)
for record in records:
    print(
        f"{record['epsilon']:10.1e} {record['condition']:14.6e} "
        f"{record['current_error']:16.6e} "
        f"{record['reference_error']:18.6e} "
        f"{record['current_residual']:18.6e} "
        f"{record['reference_residual']:20.6e}"
    )

worst_case = records[-1]
assert worst_case["current_error"] > 0.1
assert worst_case["reference_error"] < 1e-8
assert worst_case["current_error"] > worst_case["reference_error"]


### Interpreting coefficient error and residual error

As conditioning worsens, the current coefficient error grows from numerical
noise to a material fraction of the least-squares coefficient norm. The
direct-SVD reference remains close to `np.linalg.lstsq` across the same designs.

A small residual does not imply accurate coefficients here. Nearly collinear
columns permit substantially different coefficient vectors to produce similar
predictions. That is why coefficient error, rather than residual alone, exposes
the damage caused by forming `X.T @ X`.

The next check confirms that the reference is not merely an OLS replacement.
At requested fraction `0.5`, it must find a positive alpha and return a
coefficient vector whose norm is half the stable OLS norm.

In [ ]:
fraction = 0.5
design = make_tall_design(
    base_column, perturbation, extra_columns, epsilons[-1]
)
response = design @ true_coefficients
ols_coefficients = np.linalg.lstsq(design, response, rcond=None)[0]
fractional_coefficients, fractional_alpha = fracridge_direct_svd(
    design, response, fraction
)
achieved_fraction = np.linalg.norm(fractional_coefficients) / np.linalg.norm(
    ols_coefficients
)

print(f"requested fraction: {fraction:.12g}")
print(f"achieved fraction: {achieved_fraction:.12g}")
print(f"reference alpha: {fractional_alpha:.12g}")

assert np.isclose(achieved_fraction, fraction, atol=1e-10)
assert fractional_alpha > 0.0


The log-log plot summarizes the conditioning experiment. A stable method should
remain near the numerical floor as `cond(X)` increases; growth tied to
conditioning indicates that the decomposition itself is amplifying error.

In [ ]:
conditions = np.array([record["condition"] for record in records])
current_errors = np.array([record["current_error"] for record in records])
reference_errors = np.array(
    [record["reference_error"] for record in records]
)
plot_floor = np.finfo(float).eps

fig, ax = plt.subplots(figsize=(7, 4))
ax.loglog(conditions, current_errors, "o-", label="current X.T @ X path")
ax.loglog(
    conditions,
    np.maximum(reference_errors, plot_floor),
    "o-",
    label="direct economy-SVD reference",
)
ax.set(
    xlabel="condition number of X",
    ylabel="relative coefficient error",
    title="Normal equations square the conditioning problem",
)
ax.legend()
fig.tight_layout()
plt.show()


### Problem 2 acceptance criteria

The production fix should compute an economy SVD of `X` directly for tall and
wide inputs and apply rank tolerance before singular-value division. A failing
regression test must first show agreement with stable least squares at fraction
`1.0` on a controlled ill-conditioned tall design.

Additional tests must cover requested fractions below one, vector and
multi-target output shapes, rank-deficient inputs, and both JIT and non-JIT
behavior unless that implementation split is removed. Supported behavior
outside this defect should remain unchanged.

## Pull request acceptance checklist

The corrective pull request should satisfy all of the following:

- [ ] Focused intercept and numerical-stability regression tests were committed
      and observed failing before implementation.
- [ ] CV preprocessing occurs within each estimator fit, not before fold
      construction.
- [ ] A direct economy SVD of `X` replaces the normal-equations decomposition.
- [ ] Tests cover intercept recovery, coefficient accuracy, achieved fraction,
      vector and multi-target shapes, and rank-deficient behavior.
- [ ] Focused tests and the complete package suite were run before refactoring.
- [ ] Documentation is updated if public behavior or import paths change.

The executable references in this notebook establish the expected numerical
behavior. They are controls for reviewing the production patch, not package code
to copy wholesale.